# Assignment 3a

- Write a Python code to fetch data from NY Times Article Search API and push the data into the local database table.

- Create an account on https://developer.nytimes.com/

- Follow instructions from documentation and create an app and api key. (https://developer.nytimes.com/get-started)

- Api documentation link - https://developer.nytimes.com/docs/articlesearch-product/1/routes/articlesearch.json/get 

- Fetch data for all ‘Sports’ articles (parameter ‘q’ should be set to ‘sport’ in the API request) for the first 6 months of 2020 from Article Search API.

- Use pagination to get the paged data.

- Sort data using publication date.

- Fields to be Parsed -

               abstract, web_url, pub_date, ‘original’ and ‘organization’ fields from byline object (byline_original, byline_organization), _id, document_type, section_name, news_desk, 'main' field from headline object, word_count, uri.

- Load data into database.

               a. Create a table programmatically.
               b. Use try-catch blocks wherever necessary.
               c. Select unique key column(s) appropriately.
               d. Create a class and separate methods for all the functionalities like, downloading the json data, parsing the json data, db connection and insertion into table, closing the connection etc.

In [1]:
!pip install requests
!pip install pymysql

  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp311-cp311-win_amd64.whl.metadata (41 kB)
Using cached requests-2.33.1-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.7-cp311-cp311-win_amd64.whl (159 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ----------------------- 2/5 [charset_normalizer]
   ---------------- ----------------------- 2/5 [charset_n


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import pymysql
from datetime import datetime

In [5]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nyt_db"
)

cursor = conn.cursor()

print("Database Connected")

Database Connected


In [6]:
create_table_query = """

CREATE TABLE IF NOT EXISTS sports_articles (

    id VARCHAR(200) PRIMARY KEY,
    abstract TEXT,
    web_url TEXT,
    pub_date DATETIME,
    byline_original TEXT,
    byline_organization TEXT,
    document_type VARCHAR(100),
    section_name VARCHAR(100),
    news_desk VARCHAR(100),
    headline_main TEXT,
    word_count INT,
    uri TEXT
)

"""

cursor.execute(create_table_query)

conn.commit()

print("Table Created")

Table Created


In [7]:
api_key = "iLvaARGGvIZmGMoG2G3WYXJDLqwIdabTCpA5TlroN5PxDJ4S"

In [ ]:
all_articles = []

months = [
    ("20200101", "20200131"),
    ("20200201", "20200229"),
    ("20200301", "20200331"),
    ("20200401", "20200430"),
    ("20200501", "20200531"),
    ("20200601", "20200630")
]

try:

    for start_date, end_date in months:

        for page in range(0, 5):

            url = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

            params = {
                "q": "sport",
                "begin_date": start_date,
                "end_date": end_date,
                "page": page,
                "sort": "newest",
                "api-key": api_key
            }

            response = requests.get(url, params=params)

            data = response.json()

            if 'response' in data:

                docs = data['response']['docs']

                all_articles.extend(docs)

                print("Fetched Page:", page, "Date:", start_date)

            else:

                print("No response found for page", page)
                print(data)

except Exception as e:
    print("Error:", e)

Fetched Page: 0 Date: 20200101
Fetched Page: 1 Date: 20200101
Fetched Page: 2 Date: 20200101
Fetched Page: 3 Date: 20200101
Fetched Page: 4 Date: 20200101
No response found for page 0
{'fault': {'faultstring': 'Rate limit quota violation. Quota limit  exceeded. Identifier : 45d693d3-5047-4dd7-b92f-045faffaf571', 'detail': {'errorcode': 'policies.ratelimit.QuotaViolation'}}}
No response found for page 1
{'fault': {'faultstring': 'Rate limit quota violation. Quota limit  exceeded. Identifier : 45d693d3-5047-4dd7-b92f-045faffaf571', 'detail': {'errorcode': 'policies.ratelimit.QuotaViolation'}}}
No response found for page 2
{'fault': {'faultstring': 'Rate limit quota violation. Quota limit  exceeded. Identifier : 45d693d3-5047-4dd7-b92f-045faffaf571', 'detail': {'errorcode': 'policies.ratelimit.QuotaViolation'}}}
No response found for page 3
{'fault': {'faultstring': 'Rate limit quota violation. Quota limit  exceeded. Identifier : 45d693d3-5047-4dd7-b92f-045faffaf571', 'detail': {'errorcod

In [11]:
parsed_data = []

try:

    for article in all_articles:

        row = (

            article.get('_id'),

            article.get('abstract'),

            article.get('web_url'),

            article.get('pub_date'),

            article.get('byline', {}).get('original'),

            article.get('byline', {}).get('organization'),

            article.get('document_type'),

            article.get('section_name'),

            article.get('news_desk'),

            article.get('headline', {}).get('main'),

            article.get('word_count'),

            article.get('uri')

        )

        parsed_data.append(row)

    print("Data Parsed Successfully")

except Exception as e:
    print("Parsing Error:", e)

Data Parsed Successfully


In [12]:
insert_query = """

INSERT IGNORE INTO sports_articles
VALUES (%s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s, %s, %s)

"""

try:

    cursor.executemany(insert_query, parsed_data)

    conn.commit()

    print("Data Inserted Successfully")

except Exception as e:
    print("Insertion Error:", e)

Data Inserted Successfully


In [13]:
try:

    cursor.execute("SELECT * FROM sports_articles LIMIT 10")

    rows = cursor.fetchall()

    for row in rows:
        print(row)

except Exception as e:
    print("Display Error:", e)

('nyt://article/12d598c8-2127-5fe5-b4b8-f5dc8995b562', 'Florida’s freshwater wonder is threatened like never before with a rising sea level as restoration efforts lag.', 'https://www.nytimes.com/2020/01/27/travel/everglades-florida.html', datetime.datetime(2020, 1, 27, 10, 0, 22), 'By Nina Burleigh', None, 'article', 'Travel', 'Travel', 'Tears for the Magnificent and Shrinking Everglades, a ‘River of Grass’', 2639, 'nyt://article/12d598c8-2127-5fe5-b4b8-f5dc8995b562')
('nyt://article/1702d2b9-33a3-56ac-a60c-08baee24dfa0', 'A factory whose survival was a central issue in last year’s 40-day strike will get a $2.2 billion infusion for next-generation vehicles.', 'https://www.nytimes.com/2020/01/27/business/gm-detroit-electric.html', datetime.datetime(2020, 1, 27, 17, 28, 14), 'By Niraj Chokshi', None, 'article', 'Business', 'Business', 'G.M. Making Detroit Plant a Hub of Electric and A.V. Efforts', 535, 'nyt://article/1702d2b9-33a3-56ac-a60c-08baee24dfa0')
('nyt://article/1a795ba5-c3b9-52

In [14]:
cursor.execute("SELECT COUNT(*) FROM sports_articles")

count = cursor.fetchone()

print("Total Rows:", count[0])

Total Rows: 50


In [15]:
cursor.close()
conn.close()

print("Connection Closed")

Connection Closed
